In [1]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

Processing /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [2]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl

Processing /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl


# Reproducibility

In [3]:
import torch
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Paths

In [4]:
TRAIN_SEQ = "/kaggle/input/stanford-rna-3d-folding-2/train_sequences.csv"
TRAIN_LBL = "/kaggle/input/stanford-rna-3d-folding-2/train_labels.csv"

VAL_SEQ   = "/kaggle/input/stanford-rna-3d-folding-2/validation_sequences.csv"
VAL_LBL   = "/kaggle/input/stanford-rna-3d-folding-2/validation_labels.csv"

TEST_SEQ  = "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"

MSA_DIR   = "/kaggle/input/stanford-rna-3d-folding-2/MSA"
PDB_DIR   = "/kaggle/input/stanford-rna-3d-folding-2/PDB_RNA"
META_PATH = "/kaggle/input/stanford-rna-3d-folding-2/extra/rna_metadata.csv"

# Dataset (Train & Validation)

In [5]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from Bio.Seq import Seq

NUC_MAP = {'A':0, 'U':1, 'G':2, 'C':3}

def clean_sequence(seq):
    seq_obj = Seq(seq.upper())
    return "".join([n for n in str(seq_obj) if n in NUC_MAP])

def one_hot(seq):
    x = torch.zeros(len(seq), 4)
    for i, s in enumerate(seq):
        x[i, NUC_MAP[s]] = 1
    return x


class RNADataset(Dataset):

    def __init__(self, seq_csv, label_csv=None, max_length=1000):

        self.q_df = pd.read_csv(seq_csv)
        self.max_length = max_length
        self.has_labels = label_csv is not None

        if self.has_labels:
            labels = pd.read_csv(label_csv, low_memory=False)

            labels["struct_id"] = labels["ID"].str.split("_").str[0]
            labels["res_idx"]   = labels["ID"].str.split("_").str[1].astype(int)

            self.structures = {}

            # ✅ counters (NEW)
            nan_structure_count = 0
            dropped_structure_count = 0

            for k, g in labels.groupby("struct_id"):

                g = g.sort_values("res_idx")

                coords = g[["x_1", "y_1", "z_1"]].values.astype(np.float32)
                coords = torch.from_numpy(coords)

                # Remove residues with NaNs
                valid_mask = ~torch.isnan(coords).any(dim=1)
                cleaned_coords = coords[valid_mask]

                # ✅ count instead of printing
                if valid_mask.sum() != len(valid_mask):
                    nan_structure_count += 1

                # Drop structure if completely corrupted
                if cleaned_coords.shape[0] > 0:
                    self.structures[k] = cleaned_coords
                else:
                    dropped_structure_count += 1

            # ✅ single summary print
            print(f"Structures containing NaN residues: {nan_structure_count}")
            print(f"Fully corrupted structures dropped: {dropped_structure_count}")

            self.valid_ids = [
                sid for sid in self.q_df["target_id"]
                if sid in self.structures
            ]

        else:
            self.valid_ids = list(self.q_df["target_id"])

    def __len__(self):
        return len(self.valid_ids)

    def __getitem__(self, idx):

        sid = self.valid_ids[idx]
        row = self.q_df[self.q_df["target_id"] == sid].iloc[0]

        seq = clean_sequence(row["sequence"])

        if self.has_labels:

            coords = self.structures[sid]

            L = min(len(seq), coords.shape[0])
            seq = seq[:L]
            coords = coords[:L]

            # Center coordinates
            coords = coords - coords.mean(dim=0, keepdim=True)

            # Normalize scale
            coords = coords / (coords.std() + 1e-8)

            if torch.isnan(coords).any():
                raise ValueError(f"NaNs detected after normalization in {sid}")

        else:
            coords = None
            L = len(seq)

        if L > self.max_length:
            seq = seq[:self.max_length]
            if coords is not None:
                coords = coords[:self.max_length]

        x = one_hot(seq)

        # Positional feature
        pos_feat = torch.arange(len(seq)).float().unsqueeze(-1) / len(seq)
        x = torch.cat([x, pos_feat], dim=1)

        return sid, x, coords

In [6]:
dataset = RNADataset(TRAIN_SEQ, TRAIN_LBL)

print("Dataset size:", len(dataset))

Structures containing NaN residues: 3045
Fully corrupted structures dropped: 0
Dataset size: 5716


Check First Sample

In [7]:
sid, x, coords = dataset[2]

print("SID:", sid)
print("x shape:", x.shape)

SID: 1TRA
x shape: torch.Size([76, 5])


Check Alignment Integrity

In [8]:
for i in range(5):
    sid, x, coords = dataset[i]
    print(f"{sid} | x:{x.shape} | coords:{coords.shape}")
    assert x.shape[0] == coords.shape[0]

4TNA | x:torch.Size([76, 5]) | coords:torch.Size([76, 3])
6TNA | x:torch.Size([76, 5]) | coords:torch.Size([76, 3])
1TRA | x:torch.Size([76, 5]) | coords:torch.Size([76, 3])
1TN2 | x:torch.Size([76, 5]) | coords:torch.Size([76, 3])
1TN1 | x:torch.Size([76, 5]) | coords:torch.Size([76, 3])


Alignment correspondence check

In [9]:
rows = []

for i in range(10):
    sid_dataset, _, _ = dataset[i]
    
    # since you filtered valid_ids, this is safer:
    sid_from_valid_list = dataset.valid_ids[i]
    
    rows.append({
        "Index": i,
        "ID_from_valid_ids": sid_from_valid_list,
        "ID_from_dataset": sid_dataset,
        "Match": sid_from_valid_list == sid_dataset
    })

alignment_df = pd.DataFrame(rows)
alignment_df

,Index,ID_from_valid_ids,ID_from_dataset,Match
0,0,4TNA,4TNA,True
1,1,6TNA,6TNA,True
2,2,1TRA,1TRA,True
3,3,1TN2,1TN2,True
4,4,1TN1,1TN1,True
5,5,2TRA,2TRA,True
6,6,3TRA,3TRA,True
7,7,4TRA,4TRA,True
8,8,1RNA,1RNA,True
9,9,1ELH,1ELH,True


Check id alignment correspondence

In [10]:
seq_df = pd.read_csv(TRAIN_SEQ)
lbl_df = pd.read_csv(TRAIN_LBL, low_memory=False)

lbl_df["struct_id"] = lbl_df["ID"].str.split("_").str[0]

seq_ids = set(seq_df["target_id"])
label_ids = set(lbl_df["struct_id"])

print("IDs in sequences:", len(seq_ids))
print("IDs in labels:", len(label_ids))
print("Common IDs:", len(seq_ids & label_ids))
print("Missing in labels:", len(seq_ids - label_ids))

IDs in sequences: 5716
IDs in labels: 5716
Common IDs: 5716
Missing in labels: 0


Check Sequence Cleaning Actually Works

In [11]:
sid, x, coords = dataset[0]

one_hot_part = x[:, :4]
sum_per_row = one_hot_part.sum(dim=1)

print("Min row sum:", sum_per_row.min().item())
print("Max row sum:", sum_per_row.max().item())

Min row sum: 1.0
Max row sum: 1.0


Check Position Feature Range

In [12]:
pos_feature = x[:, 4]

print("Min pos:", pos_feature.min().item())
print("Max pos:", pos_feature.max().item())

Min pos: 0.0
Max pos: 0.9868420958518982


Check Coordinate Order Correct

In [13]:
import pandas as pd

labels = pd.read_csv(TRAIN_LBL, low_memory=False)
tmp = labels[labels["ID"].str.startswith(sid + "_")]

print(tmp["ID"].head())

774838    4TNA_1
774839    4TNA_2
774840    4TNA_3
774841    4TNA_4
774842    4TNA_5
Name: ID, dtype: object


 Check Max Length Cropping Works

In [14]:
small_dataset = RNADataset(TRAIN_SEQ, TRAIN_LBL, max_length=50)

# fetch first sample silently
sid, x, coords = small_dataset[0]

# print ONLY required info
print(f"Length after crop: {x.shape[0]}")

Structures containing NaN residues: 3045
Fully corrupted structures dropped: 0
Length after crop: 50


 Check Max Length Cropping Works

In [15]:
all_ids = set(dataset.q_df["target_id"])
valid_ids = set(dataset.valid_ids)

print("Missing IDs:", len(all_ids - valid_ids))

Missing IDs: 0


Structural Check

In [16]:
for i in range(20):
    sid, x, coords = dataset[i]
    if x.shape[0] != coords.shape[0]:
        print("Mismatch found:", sid)

# nothing prints = alignment stable

# Graph Builder

In [17]:
from torch_geometric.data import Data

def center_coordinates(coords):
    center = coords.mean(dim=0, keepdim=True)
    return coords - center

def build_graph(x, coords=None, k=2):

    L = x.size(0)

    edge_index = []
    edge_attr  = []

    if coords is not None:
        coords = center_coordinates(coords)

    for i in range(L):
        for j in range(max(0, i-k), min(L, i+k+1)):
            if i == j:
                continue

            edge_index.append([i, j])

            if coords is not None:
                dist = torch.norm(coords[i] - coords[j])
                edge_attr.append([dist.item()])
            else:
                edge_attr.append([abs(i-j)])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr
    )

    if coords is not None:
        data.pos = coords
        data.y   = coords

    return data

Basic Graph Build Test

In [18]:
sid, x, coords = dataset[0]

graph = build_graph(x, coords, k=2)

print("Nodes:", graph.num_nodes)
print("Edges:", graph.num_edges)
print("Node feature shape:", graph.x.shape)
print("Edge index shape:", graph.edge_index.shape)
print("Edge attr shape:", graph.edge_attr.shape)
print("Pos shape:", graph.pos.shape)

Nodes: 76
Edges: 298
Node feature shape: torch.Size([76, 5])
Edge index shape: torch.Size([2, 298])
Edge attr shape: torch.Size([298, 1])
Pos shape: torch.Size([76, 3])


Check Edge Count Logic

For k=2, each node connects to up to 4 neighbors (except boundaries).

Expected edges (approx):

𝐸 ≈𝐿×2𝑘

In [19]:
L = graph.num_nodes
print("Expected approx edges:", L * 4)
print("Actual edges:", graph.num_edges)

Expected approx edges: 304
Actual edges: 298


NaNs

In [20]:
print("NaN in x:", torch.isnan(graph.x).any().item())
print("NaN in pos:", torch.isnan(graph.pos).any().item())
print("NaN in edge_attr:", torch.isnan(graph.edge_attr).any().item())

NaN in x: False
NaN in pos: False
NaN in edge_attr: False


In [21]:
print("Coordinate mean:", graph.pos.mean(dim=0))
# not failed (checked)

Coordinate mean: tensor([-6.2742e-09, -5.4899e-09, -3.1371e-08])


# Build Train / Validation Graphs

In [22]:
train_dataset = RNADataset(TRAIN_SEQ, TRAIN_LBL)
val_dataset   = RNADataset(VAL_SEQ, VAL_LBL)
test_dataset  = RNADataset(TEST_SEQ, None)

train_graphs = []
val_graphs   = []

for sid, x, coords in train_dataset:
    train_graphs.append(build_graph(x, coords))

for sid, x, coords in val_dataset:
    val_graphs.append(build_graph(x, coords))

Structures containing NaN residues: 3045
Fully corrupted structures dropped: 0
Structures containing NaN residues: 0
Fully corrupted structures dropped: 0


Graph integrity validation

In [23]:
for i, g in enumerate(train_graphs):
    assert not torch.isnan(g.pos).any(), f"NaNs in train graph {i}"
    assert not torch.isnan(g.x).any(), f"NaNs in node features {i}"
    assert not torch.isnan(g.edge_attr).any(), f"NaNs in edge_attr {i}"

for i, g in enumerate(val_graphs):
    assert not torch.isnan(g.pos).any(), f"NaNs in val graph {i}"
    assert not torch.isnan(g.x).any(), f"NaNs in node features {i}"
    assert not torch.isnan(g.edge_attr).any(), f"NaNs in edge_attr {i}"

print("All graphs structurally clean.")

All graphs structurally clean.


Confirm Graph Counts Match Dataset

In [24]:
print("Train graphs:", len(train_graphs))
print("Train dataset:", len(train_dataset))

print("Val graphs:", len(val_graphs))
print("Val dataset:", len(val_dataset))

Train graphs: 5716
Train dataset: 5716
Val graphs: 28
Val dataset: 28


Inspect One Graph Deeply

In [25]:
g = train_graphs[0]

print("Nodes:", g.num_nodes)
print("Edges:", g.num_edges)
print("x shape:", g.x.shape)
print("pos shape:", g.pos.shape)
print("y shape:", g.y.shape)
print("edge_attr shape:", g.edge_attr.shape)

Nodes: 76
Edges: 298
x shape: torch.Size([76, 5])
pos shape: torch.Size([76, 3])
y shape: torch.Size([76, 3])
edge_attr shape: torch.Size([298, 1])


Check For Corruption Across Entire Train Set

In [26]:
for i, g in enumerate(train_graphs[:50]):
    assert g.x.shape[0] == g.pos.shape[0]
    assert g.pos.shape[0] == g.y.shape[0]
    assert g.edge_index.shape[1] == g.edge_attr.shape[0]
    assert not torch.isnan(g.x).any()
    assert not torch.isnan(g.pos).any()
    assert not torch.isnan(g.edge_attr).any()

print("Graphs structurally stable")

Graphs structurally stable


Test Batching

In [27]:
from torch_geometric.loader import DataLoader

loader = DataLoader(train_graphs, batch_size=4, shuffle=True)

batch = next(iter(loader))

print(batch)
print("Batch nodes:", batch.x.shape)
print("Batch edges:", batch.edge_index.shape)
print("Batch pos:", batch.pos.shape)

DataBatch(x=[1238, 5], edge_index=[2, 4928], edge_attr=[4928, 1], pos=[1238, 3], y=[1238, 3], batch=[1238], ptr=[5])
Batch nodes: torch.Size([1238, 5])
Batch edges: torch.Size([2, 4928])
Batch pos: torch.Size([1238, 3])


In [28]:
print("Unique graphs in batch:", batch.batch.unique())

Unique graphs in batch: tensor([0, 1, 2, 3])


Mini Forward Pass Test

In [29]:
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv

class TestModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(5, 32)
        self.conv2 = GCNConv(32, 3)

    def forward(self, data):
        x = self.conv1(data.x, data.edge_index)
        x = torch.relu(x)
        x = self.conv2(x, data.edge_index)
        return x

In [30]:
model = TestModel()

batch = next(iter(loader))
out = model(batch)

print("Output shape:", out.shape)
print("Target shape:", batch.y.shape)

Output shape: torch.Size([2100, 3])
Target shape: torch.Size([2100, 3])


Loss Test

In [31]:
loss_fn = nn.MSELoss()
loss = loss_fn(out, batch.y)

print("Loss:", loss.item())

Loss: 0.6441967487335205


Confirm the Coordinate Scale

In [32]:
print("Target min:", batch.y.min().item())
print("Target max:", batch.y.max().item())
print("Target std:", batch.y.std().item())

Target min: -3.3580517768859863
Target max: 2.371411085128784
Target std: 0.8026615977287292


In [33]:
print("Pred mean:", out.mean(dim=0))
print("Pred std:", out.std())

Pred mean: tensor([-0.0036, -0.0852,  0.0246], grad_fn=<MeanBackward1>)
Pred std: tensor(0.0756, grad_fn=<StdBackward0>)


# Device

In [34]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# DataLoaders

In [35]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(
    train_graphs,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=4,
    shuffle=False
)

# EGNN model

In [36]:
import torch.nn as nn
from torch_geometric.utils import scatter

class SimpleEGNNLayer(nn.Module):
    def __init__(self, hidden, edge_dim=1):
        super().__init__()

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden * 2 + edge_dim + 1, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(hidden + hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden, 1),
            nn.Tanh()
        )

    def forward(self, x, pos, edge_index, edge_attr):

        row, col = edge_index

        xi = x[row]
        xj = x[col]

        rel = pos[row] - pos[col]
        dist2 = (rel ** 2).sum(dim=1, keepdim=True)

        m_in = torch.cat([xi, xj, edge_attr, dist2], dim=1)

        m = self.edge_mlp(m_in)

        agg = scatter(m, row, dim=0, dim_size=x.size(0), reduce="mean")

        x = self.node_mlp(torch.cat([x, agg], dim=1))

        trans = self.coord_mlp(m) * rel
        delta = scatter(trans, row, dim=0, dim_size=pos.size(0), reduce="mean")

        pos = pos + delta

        return x, pos

In [37]:
class EGNNModel(nn.Module):

    def __init__(self, in_dim=5, hidden=64, num_layers=3):
        super().__init__()

        self.embedding = nn.Linear(in_dim, hidden)

        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(
                SimpleEGNNLayer(hidden, edge_dim=1)
            )

        self.out = nn.Linear(hidden, 3)

    def forward(self, data):

        x = data.x
        pos = data.pos
        edge_index = data.edge_index
        edge_attr = data.edge_attr

        x = self.embedding(x)

        for conv in self.convs:
            x, pos = conv(x, pos, edge_index, edge_attr)

        pred = self.out(x)

        return pred

# Model + optimizer + loss

In [38]:
model = EGNNModel().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

loss_fn = nn.MSELoss()

# Metrics

In [39]:
def mae(pred, target):
    return (pred - target).abs().mean().item()

# Train step

In [40]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    total_mae = 0.0
    n = 0

    for batch in loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        pred = model(batch)
        loss = loss_fn(pred, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_mae += mae(pred, batch.y)
        n += 1

    return total_loss / n, total_mae / n

# Validation step

In [41]:
@torch.no_grad()
def validate(model, loader):

    model.eval()

    total_loss = 0.0
    total_mae = 0.0
    n = 0

    for batch in loader:

        batch = batch.to(device)

        pred = model(batch)
        loss = loss_fn(pred, batch.y)

        total_loss += loss.item()
        total_mae += mae(pred, batch.y)
        n += 1

    return total_loss / n, total_mae / n

# Training loop

(keep this small first – you already planned 5 epochs)

In [42]:
epochs = 5

for epoch in range(epochs):

    tr_loss, tr_mae = train_one_epoch(model, train_loader)
    va_loss, va_mae = validate(model, val_loader)

    print(
        f"Epoch {epoch} | "
        f"train MSE {tr_loss:.4f} | train MAE {tr_mae:.4f} | "
        f"val MSE {va_loss:.4f} | val MAE {va_mae:.4f}"
    )

Epoch 0 | train MSE 0.7874 | train MAE 0.7077 | val MSE 1.0032 | val MAE 0.7402
Epoch 1 | train MSE 0.7877 | train MAE 0.7080 | val MSE 1.0033 | val MAE 0.7402
Epoch 2 | train MSE 0.7904 | train MAE 0.7090 | val MSE 1.0032 | val MAE 0.7402
Epoch 3 | train MSE 0.7879 | train MAE 0.7080 | val MSE 1.0035 | val MAE 0.7396
Epoch 4 | train MSE 0.7863 | train MAE 0.7075 | val MSE 1.0044 | val MAE 0.7424


# Sanity check
(the exact error you showed will not occur)

In [43]:
batch = next(iter(train_loader))
batch = batch.to(device)

out = model(batch)

print(out.shape, batch.y.shape)

torch.Size([2147, 3]) torch.Size([2147, 3])


# Build Test Graph

In [44]:
test_graphs = []

for sid, x, _ in test_dataset:

    g = build_graph(x, coords=None)

    # 🔴 IMPORTANT: give dummy initial coordinates
    g.pos = torch.zeros(g.num_nodes, 3)

    test_graphs.append(g)

# Test loader

In [45]:
test_loader = DataLoader(
    test_graphs,
    batch_size=4,
    shuffle=False
)

# Test inference

In [46]:
@torch.no_grad()
def run_test_inference(model, loader):

    model.eval()
    preds = []

    for batch in loader:
        batch = batch.to(device)
        pred = model(batch)
        preds.append(pred.cpu())

    return torch.cat(preds, dim=0)

# Run inference

In [47]:
test_predictions = run_test_inference(model, test_loader)
print(test_predictions.shape)

torch.Size([5662, 3])
